In [4]:
import pandas as pd
import numpy as np

# Másolat a tisztításhoz
df = pd.read_csv("zenga_listings_details.csv")
df_clean = df.copy()

# 1. Felesleges oszlopok törlése
df_clean = df_clean.loc[:, ~df_clean.columns.str.contains('^Unnamed')]

# 2. Mértékegységek eltávolítása és számokká alakítás
def extract_number(val):
    if pd.isna(val):
        return np.nan
    val = str(val).replace("\xa0", " ").replace(",", ".")
    # magyar millió és ezer jelölés kezelése
    if "millió" in val.lower():
        try:
            num = float(val.lower().split("millió")[0].strip())
            return num * 1_000_000
        except:
            return np.nan
    # csak számjegyek
    num_str = ''.join(ch for ch in val if ch.isdigit() or ch == '.')
    try:
        return float(num_str)
    except:
        return np.nan

numeric_cols = ["price", "area_m2", "rooms", "floors_total", "year_built", "balcony"]
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(extract_number)

# 3. Location hibás sorok javítása (ha location üres vagy furcsa, title-ből kinyerés)
def fix_location(row):
    loc = row["location"]
    if pd.isna(loc) or "leírás" in str(loc).lower():
        title = str(row["title"])
        if "," in title:
            parts = title.split(",")
            return parts[0].replace("Eladó", "").strip()
        return np.nan
    return loc.strip()

if "location" in df_clean.columns and "title" in df_clean.columns:
    df_clean["location"] = df_clean.apply(fix_location, axis=1)

# 4. Hiányzó értékek egységes kezelése
df_clean = df_clean.replace(["", " ", "nan", "None", "Nem adta meg a hirdető"], np.nan)

# 5. Szöveges oszlopok egységesítése (kisbetűsítés, whitespace eltávolítás)
text_cols = df_clean.select_dtypes(include=["object"]).columns
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Mentés
output_path = "model_ready_zenga.csv"
df_clean.to_csv(output_path, index=False)

output_path


'model_ready_zenga.csv'